# CycloSafe — Objective 2: Tropical Cyclone Intensity Estimation V2
## Deep Learning Regression with EfficientNetB0 on 39,000+ Genuine Satellite Images

This notebook trains a high-precision continuous wind-speed regression model ($V_{\max}$ in knots) on genuine true-color NASA MODIS satellite images paired with NOAA NCEI IBTrACS ground truth.

**Key Architecture Highlights:**
- **Backbone:** `EfficientNetB0` pre-trained on ImageNet (Compound Scaling, 4M parameters)
- **Task:** Direct regression predicting continuous maximum sustained wind speed (knots)
- **Data Pipeline:** Streaming `tf.data.Dataset` with parallel prefetch & augmentation
- **Evaluation:** GroupShuffleSplit by `storm_id` ensuring zero temporal data leakage across storms
- **Output Artifact:** Saves `best_cyclone_intensity_v2.keras` directly to Google Drive


## Step 1: Verify Hardware Accelerator (GPU)


In [ ]:
import tensorflow as tf
import os, sys, time

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow Version:', tf.__version__)
if gpus:
    print('✅ GPU Detected:', gpus)
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
else:
    print('⚠️ WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> Select T4 GPU.')


## Step 2: Mount Google Drive & Extract Dataset


In [ ]:
from google.colab import drive
import zipfile, shutil

drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/cyclosafe_intensity_dataset.zip'
EXTRACT_DIR = '/content/dataset'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"Could not find {ZIP_PATH}! Please upload 'cyclosafe_intensity_dataset.zip' to your Google Drive root.")

if not os.path.exists(EXTRACT_DIR):
    print(f"Extracting {ZIP_PATH} to high-speed Colab NVMe SSD ({EXTRACT_DIR})...")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("✅ Extraction complete!")
else:
    print(f"Dataset already extracted at {EXTRACT_DIR}")

# Verify contents
images_dir = os.path.join(EXTRACT_DIR, 'images')
labels_csv = os.path.join(EXTRACT_DIR, 'metadata', 'labels.csv')
num_images = len([f for f in os.listdir(images_dir) if f.endswith('.jpg')])
print(f"Verified dataset images: {num_images:,}")
print(f"Labels CSV present: {os.path.exists(labels_csv)}")


## Step 3: Load Metadata & Group-Split by Storm ID (Zero Leakage)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
import matplotlib.pyplot as plt

df = pd.read_csv(labels_csv)
print(f"Total metadata records: {len(df):,}")

# Filter valid wind speeds and existing image paths
df['full_image_path'] = df['image_path'].apply(lambda p: os.path.join(EXTRACT_DIR, p))
df = df[df['full_image_path'].apply(os.path.exists)].copy()
df['wind_speed_knots'] = df['wind_speed_knots'].astype(float)
df = df[df['wind_speed_knots'] > 0].copy()

print(f"Valid verified image-label pairs: {len(df):,}")
print(f"Unique tropical cyclones: {df['storm_id'].nunique():,}")
print(f"Wind speed range: {df['wind_speed_knots'].min():.1f} kts to {df['wind_speed_knots'].max():.1f} kts (Mean: {df['wind_speed_knots'].mean():.1f} kts)")

# Strict group split: storms in Train NEVER appear in Validation or Test
gss = GroupShuffleSplit(n_splits=1, train_size=0.85, random_state=42)
train_idx, temp_idx = next(gss.split(df, groups=df['storm_id']))

train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

gss_val = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df['storm_id']))

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

print(f"\nDataset Partitioning:")
print(f"  - Training Set:   {len(train_df):,} images ({train_df['storm_id'].nunique()} storms)")
print(f"  - Validation Set: {len(val_df):,} images ({val_df['storm_id'].nunique()} storms)")
print(f"  - Test Set:       {len(test_df):,} images ({test_df['storm_id'].nunique()} storms)")


## Step 4: High-Performance tf.data Streaming Pipeline


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def parse_image(path, label):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0  # Normalize to [0, 1]
    return img, tf.cast(label, tf.float32)

def augment_image(img, label):
    # Cyclones rotate in both hemispheres; horizontal/vertical flips and rotations are physically valid
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.08)
    img = tf.image.random_contrast(img, lower=0.92, upper=1.08)
    return img, label

def create_dataset(dataframe, is_training=False):
    paths = dataframe['full_image_path'].values
    labels = dataframe['wind_speed_knots'].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        ds = ds.shuffle(buffer_size=4096)
        ds = ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

train_ds = create_dataset(train_df, is_training=True)
val_ds = create_dataset(val_df, is_training=False)
test_ds = create_dataset(test_df, is_training=False)
print("✅ High-throughput data pipelines initialized with parallel prefetching.")


## Step 5: Build EfficientNetB0 Regression Model


In [ ]:
from tensorflow.keras import layers, models, optimizers, callbacks, losses

def build_cyclone_intensity_model():
    base_model = tf.keras.applications.EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
    )
    base_model.trainable = False  # Freeze during Phase 1
    
    inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="satellite_input")
    x = base_model(inputs)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(128, activation="relu", name="dense_128")(x)
    x = layers.Dropout(0.3, name="dropout_0.3")(x)
    outputs = layers.Dense(1, activation="linear", name="wind_speed_knots")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="EfficientNetB0_Intensity_V2")
    return model, base_model

model, base_model = build_cyclone_intensity_model()
model.summary()

# Use Huber Loss (Smooth L1) for robustness against wind-speed sensor noise/outliers
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss=losses.Huber(delta=10.0),
    metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
)


## Step 6: Phase 1 — Warmup Head Training (Base Model Frozen)


In [ ]:
MODEL_SAVE_PATH = '/content/drive/MyDrive/best_cyclone_intensity_v2.keras'

cb_phase1 = [
    callbacks.EarlyStopping(monitor='val_mae', patience=3, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=2, min_lr=1e-5, verbose=1)
]

print("Starting Phase 1: Training regression head (5 epochs)...")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=cb_phase1
)


## Step 7: Phase 2 — End-to-End Fine-Tuning (Top 40 Layers Unfrozen)


In [ ]:
base_model.trainable = True
# Fine-tune the top 40 layers while keeping lower low-level edge filters frozen
for layer in base_model.layers[:-40]:
    layer.trainable = False

print(f"Total layers: {len(base_model.layers)}, Trainable layers: {sum([1 for l in base_model.layers if l.trainable])}")

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),  # Lower learning rate for fine-tuning
    loss=losses.Huber(delta=10.0),
    metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
)

cb_phase2 = [
    callbacks.ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_mae', mode='min', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor='val_mae', patience=5, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

print("Starting Phase 2: Fine-tuning EfficientNetB0 (15 epochs)...")
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=cb_phase2
)
print(f"✅ Training completed! Best model saved to: {MODEL_SAVE_PATH}")


## Step 8: Comprehensive Model Evaluation on Unseen Test Storms


In [ ]:
print(f"Evaluating best model on {len(test_df):,} held-out test observations...")
test_metrics = model.evaluate(test_ds)
print(f"Test Loss (Huber): {test_metrics[0]:.3f}")
print(f"Test Mean Absolute Error (MAE): {test_metrics[1]:.2f} knots")
print(f"Test Root Mean Squared Error (RMSE): {test_metrics[2]:.2f} knots")

# Generate Predictions vs Ground Truth scatter plot
y_true = []
y_pred = []
for batch_x, batch_y in test_ds:
    preds = model.predict(batch_x, verbose=0).flatten()
    y_pred.extend(preds)
    y_true.extend(batch_y.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

plt.figure(figsize=(8, 8))
plt.scatter(y_true, y_pred, alpha=0.3, s=15, color='#0284c7', label='Observations')
plt.plot([0, 180], [0, 180], 'r--', lw=2, label='Ideal Equality (y=x)')
plt.xlabel('Ground Truth Wind Speed (knots)', fontsize=12)
plt.ylabel('EfficientNetB0 Predicted Wind Speed (knots)', fontsize=12)
plt.title(f'CycloSafe Intensity Estimation Test Performance\nMAE: {test_metrics[1]:.2f} kts | RMSE: {test_metrics[2]:.2f} kts', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.xlim(0, 180)
plt.ylim(0, 180)
plt.show()


## Step 9: Download Trained Model for CycloSafe Backend
Once training completes, your new model is saved at:
`/content/drive/MyDrive/best_cyclone_intensity_v2.keras`

**To use in CycloSafe:**
1. Download `best_cyclone_intensity_v2.keras` from Google Drive to your PC.
2. Place it into `backend/models/best_cyclone_intensity_v2.keras`.
3. Your backend API will immediately load and serve high-precision intensity predictions!
